In [0]:
%sql
USE CATALOG credlake;
USE SCHEMA bronze;

SELECT
    current_catalog() AS current_catalog,
    current_schema() AS current_schema;

In [0]:
%sql
CREATE TABLE IF NOT EXISTS credlake.bronze.products_raw (
    product_id                         INT,
    product_name                       STRING,
    allowed_customer_type              STRING,
    base_annual_interest_rate          DECIMAL(10,4),
    max_term_months                    INT,
    source_batch_id                    STRING,
    source_file_path                   STRING,
    source_file_name                   STRING,
    source_file_modification_time      TIMESTAMP,
    ingested_at                        TIMESTAMP
)
USING DELTA
COMMENT 'Produtos financeiros recebidos da origem, sem aplicação de regras de qualidade.'
TBLPROPERTIES (
    'data_layer' = 'bronze',
    'data_domain' = 'credit',
    'contains_synthetic_data' = 'true'
);


CREATE TABLE IF NOT EXISTS credlake.bronze.contracts_raw (
    contract_id                        BIGINT,
    customer_id                        BIGINT,
    product_id                         INT,
    contract_date                      DATE,
    principal_amount                   DECIMAL(18,2),
    annual_interest_rate               DECIMAL(10,4),
    term_months                        INT,
    contract_status                    STRING,
    source_updated_at                  TIMESTAMP,
    source_system                      STRING,
    source_batch_id                    STRING,
    source_file_path                   STRING,
    source_file_name                   STRING,
    source_file_modification_time      TIMESTAMP,
    ingested_at                        TIMESTAMP
)
USING DELTA
COMMENT 'Contratos financeiros recebidos da origem, incluindo registros inválidos para tratamento posterior.'
TBLPROPERTIES (
    'data_layer' = 'bronze',
    'data_domain' = 'credit',
    'contains_synthetic_data' = 'true'
);

In [0]:
%sql
-- LEITURA DOS ARQUIVOS CSV e INSERÇÃO
SELECT
    product_id,
    product_name,
    allowed_customer_type,
    base_annual_interest_rate,
    max_term_months,
    source_batch_id,
    _metadata.file_name AS source_file_name
FROM read_files(
    '/Volumes/credlake/landing/raw/products/full/',
    format => 'csv',
    header => true
)
LIMIT 10;

SELECT
    contract_id,
    customer_id,
    product_id,
    contract_date,
    principal_amount,
    annual_interest_rate,
    term_months,
    contract_status,
    source_updated_at,
    source_system,
    source_batch_id,
    _metadata.file_name AS source_file_name
FROM read_files(
    '/Volumes/credlake/landing/raw/contracts/',
    format => 'csv',
    header => true
)
LIMIT 10;

In [0]:
%sql
COPY INTO credlake.bronze.products_raw
FROM (
    SELECT
        CAST(product_id AS INT)                         AS product_id,
        CAST(product_name AS STRING)                    AS product_name,
        CAST(allowed_customer_type AS STRING)           AS allowed_customer_type,
        CAST(base_annual_interest_rate AS DECIMAL(10,4)) AS base_annual_interest_rate,
        CAST(max_term_months AS INT)                    AS max_term_months,
        CAST(source_batch_id AS STRING)                 AS source_batch_id,
        _metadata.file_path                             AS source_file_path,
        _metadata.file_name                             AS source_file_name,
        _metadata.file_modification_time                AS source_file_modification_time,
        current_timestamp()                             AS ingested_at
    FROM '/Volumes/credlake/landing/raw/products/full/'
)
FILEFORMAT = CSV
FORMAT_OPTIONS (
    'header' = 'true'
)
COPY_OPTIONS (
    'mergeSchema' = 'false'
);

In [0]:
%sql
COPY INTO credlake.bronze.contracts_raw
FROM (
    SELECT
        CAST(contract_id AS BIGINT)                     AS contract_id,
        CAST(customer_id AS BIGINT)                     AS customer_id,
        CAST(product_id AS INT)                         AS product_id,
        CAST(contract_date AS DATE)                     AS contract_date,
        CAST(principal_amount AS DECIMAL(18,2))         AS principal_amount,
        CAST(annual_interest_rate AS DECIMAL(10,4))     AS annual_interest_rate,
        CAST(term_months AS INT)                        AS term_months,
        CAST(contract_status AS STRING)                 AS contract_status,
        CAST(source_updated_at AS TIMESTAMP)            AS source_updated_at,
        CAST(source_system AS STRING)                   AS source_system,
        CAST(source_batch_id AS STRING)                 AS source_batch_id,
        _metadata.file_path                             AS source_file_path,
        _metadata.file_name                             AS source_file_name,
        _metadata.file_modification_time                AS source_file_modification_time,
        current_timestamp()                             AS ingested_at
    FROM '/Volumes/credlake/landing/raw/contracts/'
)
FILEFORMAT = CSV
FORMAT_OPTIONS (
    'header' = 'true'
)
COPY_OPTIONS (
    'mergeSchema' = 'false'
);

In [0]:
%sql
SELECT
    'products_raw' AS dataset,
    COUNT(*) AS total_rows,
    COUNT(DISTINCT source_file_path) AS source_files,
    MIN(ingested_at) AS first_ingestion,
    MAX(ingested_at) AS last_ingestion
FROM credlake.bronze.products_raw

UNION ALL

SELECT
    'contracts_raw' AS dataset,
    COUNT(*) AS total_rows,
    COUNT(DISTINCT source_file_path) AS source_files,
    MIN(ingested_at) AS first_ingestion,
    MAX(ingested_at) AS last_ingestion
FROM credlake.bronze.contracts_raw;

**Exploração de inconsistências de dados:**

In [0]:
%sql
SELECT
    contract_id,
    COUNT(*) AS occurrences
FROM credlake.bronze.contracts_raw
GROUP BY contract_id
HAVING COUNT(*) > 1
ORDER BY occurrences DESC;

In [0]:
%sql
SELECT
    SUM(CASE WHEN principal_amount <= 0 THEN 1 ELSE 0 END)
        AS contracts_with_invalid_amount,

    SUM(CASE WHEN customer_id = 9999999 THEN 1 ELSE 0 END)
        AS contracts_with_orphan_customer,

    SUM(CASE WHEN contract_id = 9000001 THEN 1 ELSE 0 END)
        AS known_negative_contract,

    SUM(CASE WHEN contract_id = 9000002 THEN 1 ELSE 0 END)
        AS known_orphan_contract
FROM credlake.bronze.contracts_raw;

In [0]:
%sql
DESCRIBE HISTORY credlake.bronze.products_raw;
DESCRIBE HISTORY credlake.bronze.contracts_raw;